# 代码的内存消耗

[参考文章](https://programmercarl.com/algo/fundamentals/memory-usage.html)

## 概念

实际内存占用与语言实现、对象结构、容器容量和运行环境有关，同样是线性空间，不同程序占用的字节数也可能相差很多

原文从 C++ 的存储区域和内存对齐展开；在 Python 中，不能直接把“元素数量 × C++ 整数大小”作为列表占用的内存

## Python 对象与引用

Python 变量保存对象的引用。普通 list 保存的是元素引用，而不是把所有整数对象本身紧密放在列表内部

对象本身还可能有类型、管理信息等开销；列表通常也会预留一定容量，以支持后续追加

### 赋值与复制


In [ ]:
nums = [1, 2, 3]
alias = nums
copied = nums[:]

print(alias is nums) # True，两者指向同一个列表
print(copied is nums) # False，切片创建了新列表

alias[0] = 99
print(nums) # [99, 2, 3]
print(copied) # [1, 2, 3]


`alias = nums` 不会复制全部元素；切片是浅拷贝，创建新列表，但其中的嵌套对象仍可能共享

### 观察对象直接占用的字节数

`sys.getsizeof()` 返回对象本身直接占用的大小，不会递归统计它引用的全部对象，也不等于整个进程的内存


In [ ]:
import sys

values = [0] * 1000
print('列表本身:', sys.getsizeof(values))
print('一个整数对象:', sys.getsizeof(values[0]))
print('所有元素是否引用同一个对象:', all(value is values[0] for value in values))

small_integer = 1
large_integer = 1 << 1000
print('较小整数:', sys.getsizeof(small_integer))
print('较大整数:', sys.getsizeof(large_integer))


上面 `[0] * 1000` 重复的是同一个对象引用，不能把这个整数的对象大小重复计入 1000 次来求独立对象的总占用

输出字节数会受 Python 实现和平台影响，不需要背某个固定值。大整数位数增长时，其对象占用也会增加

### 二维列表的共享问题


In [ ]:
shared_rows = [[0] * 3] * 2
shared_rows[0][0] = 1
print(shared_rows) # [[1, 0, 0], [1, 0, 0]]

independent_rows = [[0] * 3 for _ in range(2)]
independent_rows[0][0] = 1
print(independent_rows) # [[1, 0, 0], [0, 0, 0]]


前者重复了同一个行列表的引用；后者每次循环创建一个新的行列表

## 分析方法

- 先列出主要容器和最大规模，判断是常数、线性还是平方级增长
- 分清新建对象、共享引用和浅拷贝，避免漏算或重复计算
- 算入递归栈、排序临时空间，以及同一时刻保留的中间结果
- 判断真实内存是否够用时，还要留出解释器和对象管理的开销

节点从链表中断开后，如果还有其他变量引用它，对象仍然可以存在；删除一个变量名也不等于立即将进程内存归还给操作系统

[Python 文档：sys.getsizeof](https://docs.python.org/3.11/library/sys.html#sys.getsizeof)
